# First solve

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimlab-wustl/Nimbus/blob/main/docs/tutorials/01_first_solve.ipynb)

Solve a small constrained problem with NIMBUS, read the results, and look inside the compiled machine.
The problem: select exactly 3 of 6 spins so that the energy, which rewards two triangles of spins, is lowest.
Its optimum has energy −7.

In [1]:
# In Google Colab, fetch the repository and install the package; locally, run from the repository.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("Nimbus"):
    !git clone -q https://github.com/aimlab-wustl/Nimbus.git
    !pip install -q -e Nimbus
if os.path.exists("Nimbus"):
    os.chdir("Nimbus")
elif os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("../..")                                   # docs/tutorials -> repository root

## 1. Load and run a configuration

A run is described by one YAML configuration: the problem, its move library, the annealing schedule, the
replicas and the backend. Fields can be overridden from Python.

In [2]:
from nimbus import api

cfg = api.load_config("examples/toy_pair_exchanges.yaml",
                      {"run.replicas": 16, "run.targets": -7, "output.overwrite": True})
outcome = api.solve(cfg, output_dir="local_runs/notebook_toy")
print(outcome.status)

ok


## 2. Read the results

Every replica gives one row: its lowest feasible energy and the iteration at which it first reached the
target.

In [3]:
import pandas as pd

rows = pd.read_parquet(f"{outcome.run_dir}/results.parquet")
rows[["replica_id", "best_feasible_objective", "target_hit", "target_hit_attempt"]].head(8)

,replica_id,best_feasible_objective,target_hit,target_hit_attempt
0,0,-7.0,True,51
1,1,-7.0,True,103
2,2,-7.0,True,50
3,3,-7.0,True,11
4,4,-7.0,True,12
5,5,-7.0,True,27
6,6,-7.0,True,59
7,7,-7.0,True,73


In [4]:
import math

B = int(rows["attempts"].max())
p = rows["target_hit"].mean()
its = B * math.log(0.01) / math.log(1 - p) if 0 < p < 1 else (B if p == 1 else float("inf"))
print(f"success fraction {p:.2f} at a budget of {B} iterations")
print(f"iterations to solution (99%): {its:.0f}" if p < 1 else "every replica reached the optimum")

success fraction 1.00 at a budget of 400 iterations
every replica reached the optimum


## 3. Look inside the compiled machine

The compiler turns the objective and the moves into stored clauses (products of spins), one move neuron per
move, and the switched connections between them.

In [5]:
problem, model, timings = api.compile_problem("examples/toy_pair_exchanges.yaml")
d = model.describe()
print("spins", d["n_spins"], "| stored clauses", d["n_clauses"], "of which auxiliary", d["M_add"],
      "| move neurons", d["n_groups"], "| connections", d["n_routes"])
print("moves:", [str(x) for x in model.group_label[:6]], "...")

spins 6 | stored clauses 23 of which auxiliary 9 | move neurons 15 | connections 168
moves: ['swap(0,1)', 'swap(0,2)', 'swap(0,3)', 'swap(0,4)', 'swap(0,5)', 'swap(1,2)'] ...


Every switch is fixed or set by one stored clause: `route_ctrl` names the controlling clause of each
connection (−1 for a fixed switch).

In [6]:
import numpy as np

g = 0                                   # the first move neuron
a, b = model.route_ptr[g], model.route_ptr[g + 1]
for i in range(a, b):
    k, c = model.route_clause[i], model.route_ctrl[i]
    clause = model.clause_spins[model.clause_ptr[k]:model.clause_ptr[k + 1]].tolist()
    ctrl = model.clause_spins[model.clause_ptr[c]:model.clause_ptr[c + 1]].tolist() if c >= 0 else "fixed"
    print(f"connection to clause {clause!s:10} switch set by clause {ctrl}")

connection to clause [0]        switch set by clause [0, 1]
connection to clause [0, 2]     switch set by clause [0, 1]
connection to clause [1]        switch set by clause [0, 1]
connection to clause [1, 2]     switch set by clause [0, 1]
connection to clause [0, 3]     switch set by clause [0, 1]
connection to clause [0, 4]     switch set by clause [0, 1]
connection to clause [0, 5]     switch set by clause [0, 1]
connection to clause [1, 3]     switch set by clause [0, 1]
connection to clause [1, 4]     switch set by clause [0, 1]
connection to clause [1, 5]     switch set by clause [0, 1]


## Next

* [Your own problem](https://aimlab-wustl.github.io/Nimbus/tutorials/02_your_own_problem/): write an objective and constraints and solve them.
* The documentation: https://aimlab-wustl.github.io/Nimbus